# User Management Demo

This notebook demonstrates how to manage user access and roles for both ETEngine scenarios (Sessions) and MyETM saved scenarios using pyetm.

## Prerequisites

- You need to be authenticated with a valid access token
- You must be an owner of the scenario to manage users
- The scenarios must exist before you can add users to them

In [ ]:
# Check the environment is properly configured.
from example_helpers import setup_notebook
setup_notebook()

## Part 1: Managing Users on ETEngine Sessions

Sessions are temporary working scenarios in ETEngine. You can share these with collaborators.

### Create or Load a Session

In [ ]:
from pyetm.models.scenario import Scenario
from pyetm.models.session import Session

# Option 1: Create a new session
session = Session.new(
    area_code="nl",
    end_year=2050,
    title="User Management Demo Session"
)
print(f"Created session with ID: {session.id}")

# Option 2: Load an existing session
# session = Session.load(123)  # Replace with your scenario ID

### List Current Users

In [ ]:
# Fetch all users with access to this session
users = session.list_users()

print(f"Current users ({len(users)}):")
for user in users:
    email = user.get('user_email') or f"User ID {user.get('user_id')}"
    role = user.get('role')
    print(f"  - {email}: {role}")

### Add a Viewer

Viewers have read-only access to the scenario.

In [ ]:
# Add a viewer (read-only access)
session.update_users("viewer@example.com", "scenario_viewer")
print("Added viewer@example.com as viewer")

# Verify the user was added
users = session.list_users()
print(f"Total users: {len(users)}")

### Add a Collaborator

Collaborators can modify the scenario but cannot manage users.

In [ ]:
# Add a collaborator (can modify scenario)
session.update_users("collaborator@example.com", "scenario_collaborator")
print("Added collaborator@example.com as collaborator")

# Verify the user was added
users = session.list_users()
print(f"Total users: {len(users)}")

### Add Another Owner

Owners have full control, including user management.

In [ ]:
# Add another owner (full control)
session.update_users("owner@example.com", "scenario_owner")
print("Added owner@example.com as owner")

# Verify the user was added
users = session.list_users()
print(f"Total users: {len(users)}")

### Update a User's Role

You can promote or demote users by calling update_users with the same email and a different role.

In [ ]:
# Promote viewer to collaborator
session.update_users("viewer@example.com", "scenario_collaborator")
print("Promoted viewer@example.com to collaborator")

# Verify the change
users = session.list_users()
viewer = next((u for u in users if u.get('user_email') == 'viewer@example.com'), None)
if viewer:
    print(f"viewer@example.com is now: {viewer.get('role')}")

### Remove a User

To remove a user's access, use the "remove" role.

In [ ]:
# Remove a user
session.update_users("collaborator@example.com", "remove")
print("Removed collaborator@example.com")

# Verify the user was removed
users = session.list_users()
print(f"Remaining users: {len(users)}")

In [ ]:
# Fetch all users with access to this session
users = session.list_users()

print(f"Current users ({len(users)}):")
for user in users:
    email_or_id = user.get('user_email') or f"User ID {user.get('user_id')}"
    role = user.get('role')
    print(f"  - {email_or_id}: {role}")

## Part 2: Managing Users on Saved Scenarios

Saved scenarios are persistent snapshots stored in MyETM. User management on saved scenarios also grants access to all associated ETEngine sessions.

### Save Your Session

In [ ]:
# Save the session as a saved scenario
scenario = session.save(
    title="User Management Demo - Saved",
    description="Demonstrating user management on saved scenarios",
    private=False
)
print(f"Created saved scenario with ID: {scenario.id}")

# Option 2: Load an existing saved scenario
# scenario = Scenario.load(456)  # Replace with your saved scenario ID

### List Current Users on Saved Scenario

In [ ]:
# Fetch all users with access to this saved scenario
users = scenario.list_users()

print(f"Current users on saved scenario ({len(users)}):")
for user in users:
    email = user.get('user_email') or f"User ID {user.get('user_id')}"
    role = user.get('role')
    print(f"  - {email}: {role}")

### Add Users to Saved Scenario

When you add a user to a saved scenario:
- The user is **immediately** added to the SavedScenario in MyETM
- The user is added to the current and all historical ETEngine scenarios via a background job (asynchronous)

This means there is a brief delay (typically a few seconds) before the user has access to the underlying ETEngine scenarios. This asynchronous approach prevents circular dependencies between MyETM and ETEngine.

In [ ]:
# Add users with different roles
scenario.update_users("viewer@saved.com", "scenario_viewer")
scenario.update_users("collaborator@saved.com", "scenario_collaborator")
scenario.update_users("owner@saved.com", "scenario_owner")

print("Added 3 users to saved scenario")

# Verify
users = scenario.list_users()
print(f"Total users: {len(users)}")

### Update and Remove Users

In [ ]:
# Promote viewer to collaborator
scenario.update_users("viewer@saved.com", "scenario_collaborator")
print("Promoted viewer@saved.com to collaborator")

# Remove a user
scenario.update_users("owner@saved.com", "remove")
print("Removed owner@saved.com")

# Final user list
users = scenario.list_users()
print(f"\nFinal user list ({len(users)}):")
for user in users:
    email = user.get('user_email') or f"User ID {user.get('user_id')}"
    role = user.get('role')
    print(f"  - {email}: {role}")